In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("News Data Analysis") \
    .getOrCreate()

# Define the schema manually
schema = StructType([
    StructField("timestamp", IntegerType(), True),
    StructField("source", StringType(), True),
    StructField("archive", StringType(), True),
    StructField("id", IntegerType(), True),
    StructField("probability", FloatType(), True),
    StructField("keywords", MapType(StringType(), IntegerType()), True),
    StructField("sentiment", FloatType(), True),
    #StructField("status", StringType(), True),
    #StructField("error", StringType(), True)
])

# Load the DataFrame with the defined schema
df = spark.read.format("json").schema(schema).load("data/news/status=success")

In [ ]:
length = df.count()
files_loaded = df.inputFiles()
print(f"Number of rows in the DataFrame: {length}")
print(f"Files loaded by Spark: {len(files_loaded)}")

df.printSchema()

In [ ]:
df.show()

In [ ]:
from pyspark.sql import functions as F

keyword = "Galp"

# Check if the key "X" exists in the "keywords" map
#df_with_key = df.filter(df["keywords"].getItem(keyword).isNotNull())
df_with_key = df.filter(F.col("keywords").getItem(keyword).isNotNull() & (F.col("keywords").getItem(keyword) > 4))

length = df_with_key.count()
print(f"Number of rows in the DataFrame: {length}")

df_with_key.show(truncate=False)

In [ ]:
# Process data
result = (
    df_with_key.rdd
    .flatMap(lambda row: [
        (key, (value,
               {row["timestamp"]: value},
               row["sentiment"]*value,
               {row["source"]: 1},
               [row["archive"]])) for key, value in row["keywords"].items()
    ])
    .reduceByKey(lambda a, b: (
        a[0] + b[0],  # Sum count values
        {ts: a[1].get(ts, 0) + b[1].get(ts, 0) for ts in set(a[1]) | set(b[1])},  # Merge timestamp counts
        a[2] + b[2],  # Sum sentiment values
        {source: a[3].get(source, 0) + b[3].get(source, 0) for source in set(a[3]) | set(b[3])},  # Merge source counts
        a[4] + b[4]  # Merge archive lists
    ))
    .collect()
)

# Convert to dictionary
output = {key: {"count": value[0],
                "date": value[1],
                "sentiment": value[2]/value[0],
                "source": value[3],
                "news": value[4]} for key, value in result}

# Print output
#output

sorted_dict = dict(sorted(output.items(), key=lambda item: item[1]["count"], reverse=True))
#sorted_dict


In [ ]:
import json

In [ ]:
file_path = 'data/keywords_test.json'

# Save the dictionary to a JSON file
with open(file_path, 'w') as json_file:
    json.dump(output, json_file, indent=4)

print(f"JSON data saved to {file_path}")

# Load the JSON data back into a Python dictionary
with open(file_path, 'r') as json_file:
    loaded_data = json.load(json_file)

# Print loaded data
print("Loaded data:")
#loaded_data

In [ ]:
#loaded_data["Comerciantes"]["news"][2]